# Titanic Survival Analysis - Feature Engineering

In this notebook, we prepare the dataset for machine learning by transforming and creating features that improve model performance.

This includes encoding categorical variables, creating new features, and refining the dataset.

  Goals:

- Convert categorical variables into numerical format  
- Create new meaningful features  
- Improve predictive power of the dataset  
- Prepare data for modeling  

# Import libraries

In [1]:
import pandas as pd
import numpy as np

# Load dataset

In [3]:
df = pd.read_csv("../data/processed/cleaned_titanic.csv")

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S


##  Data overview

We briefly inspect the dataset before applying transformations.

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          891 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Embarked     891 non-null    object 
dtypes: float64(2), int64(5), object(4)
memory usage: 76.7+ KB


###  Encoding sex

We convert the 'Sex' column into numerical values to make it usable for machine learning models.

In [5]:
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

###  Encoding embarked

We encode the 'Embarked' column using one-hot encoding to represent categorical values numerically.

The 'Embarked' feature is included as it may capture geographical or socioeconomic patterns that could influence survival.

In [6]:
df['Embarked'].isnull().sum()

np.int64(0)

In [7]:
df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)

###  Family size

We create a new feature representing the total number of family members aboard.

Family size may influence survival, as passengers traveling in small groups could have had better coordination and support, while large families might have faced difficulties during evacuation.

In [8]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

###  Is alone

We create a feature indicating whether a passenger was traveling alone.

In [9]:
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

###  Title extraction

We extract titles from passenger names to capture social status and potential survival patterns.
- Mr. → adult male  
- Mrs. → married woman  
- Miss. → young woman  
- Master. → child  
- Dr., Sir, Lady → social status  

In [10]:
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

<>:1: SyntaxWarning: invalid escape sequence '\.'
<>:1: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_1723/172848211.py:1: SyntaxWarning: invalid escape sequence '\.'
  df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)


In [11]:
df['Title'] = df['Title'].replace(
    ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major',
     'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare'
)

df['Title'] = df['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

In [12]:
df = pd.get_dummies(df, columns=['Title'], drop_first=True)

###  Dropping unnecessary columns

We remove columns that are not useful for modeling.

In [13]:
df = df.drop(columns=['Name', 'Ticket', 'Cabin'], errors='ignore')

### Saving processed dataset

We save the final dataset to be used in the modeling phase.

In [15]:
df.to_csv("../data/processed/titanic_featured.csv", index=False)